# Sound Timeline Agent

Run the sound timeline agent on the tracked demo asset, save the completed `VideoAsset`, and render the final `SoundTimeline`.

In [6]:
from collections import Counter
from pathlib import Path

import dotenv

dotenv.load_dotenv(dotenv.find_dotenv())

from v2a_inspect.agents.sound_timeline import run_sound_timeline_agent
from v2a_inspect.models import VideoAsset
from v2a_inspect.visualization import (
    display_image,
    display_video,
    render_sound_timeline,
    summarize_sound_timeline,
)

## Config

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_ASSET_PATH = PROJECT_ROOT / "demo" / "outputs" / "event_track_extraction" / "video_asset.with_events.json"
OUTPUT_DIR = PROJECT_ROOT / "demo" / "outputs" / "sound_timeline_agent"
OUTPUT_ASSET_PATH = OUTPUT_DIR / "video_asset.with_sound_timeline.json"

RUN_AGENT = True
MAX_ITERATIONS = None
OBJECTIVE = "Build a complete SoundTimeline for the full video."

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "input_asset_path": str(INPUT_ASSET_PATH),
    "output_asset_path": str(OUTPUT_ASSET_PATH),
    "run_agent": RUN_AGENT,
    "max_iterations": MAX_ITERATIONS,
}

## Load Input

In [ ]:
input_video_asset = VideoAsset.model_validate_json(INPUT_ASSET_PATH.read_text(encoding="utf-8"))
track_count = sum(len(scene.scene_tracks) for scene in input_video_asset.initial_scenes)

{
    "source_path": str(input_video_asset.source_path),
    "frame_count": input_video_asset.frame_count,
    "duration_sec": input_video_asset.duration_sec,
    "scene_count": len(input_video_asset.initial_scenes),
    "track_count": track_count,
    "has_visual_identity_layer": input_video_asset.visual_identity_layer is not None,
    "visual_event_count": 0
    if input_video_asset.visual_identity_layer is None
    else len(input_video_asset.visual_identity_layer.visual_events),
    "has_sound_timeline": input_video_asset.sound_timeline is not None,
}

In [ ]:
display_video(input_video_asset.source_path)

## Run Agent

In [ ]:
if RUN_AGENT:
    video_asset = input_video_asset
    run_sound_timeline_agent(
        video_asset,
        objective=OBJECTIVE,
        max_iterations=MAX_ITERATIONS,
    )
    OUTPUT_ASSET_PATH.write_text(
        video_asset.model_dump_json(indent=2, exclude_computed_fields=True),
        encoding="utf-8",
    )
elif OUTPUT_ASSET_PATH.exists():
    video_asset = VideoAsset.model_validate_json(OUTPUT_ASSET_PATH.read_text(encoding="utf-8"))
else:
    video_asset = input_video_asset
    print("Skipped. Set RUN_AGENT = True to create a SoundTimeline.")

OUTPUT_ASSET_PATH

## Output Summary

In [ ]:
sound_timeline = video_asset.sound_timeline
if sound_timeline is None:
    raise ValueError("Run the agent or load an asset with sound_timeline first")

track_type_counts = Counter(event.track_type for event in sound_timeline.sound_events)
generation_mode_counts = Counter(event.generation_mode for event in sound_timeline.sound_events)

{
    "sound_source_count": len(sound_timeline.sound_sources),
    "sound_event_count": len(sound_timeline.sound_events),
    "track_type_counts": dict(sorted(track_type_counts.items())),
    "generation_mode_counts": dict(sorted(generation_mode_counts.items())),
}

## Render Completed Timeline

In [ ]:
timeline_image = render_sound_timeline(video_asset)
timeline_image_path = OUTPUT_DIR / "sound_timeline.png"
timeline_image.save(timeline_image_path)
display_image(timeline_image)
timeline_image_path

In [ ]:
summarize_sound_timeline(video_asset)

## Outputs

In [ ]:
{
    "completed_asset": str(OUTPUT_ASSET_PATH),
    "timeline_image": str(timeline_image_path),
    "langfuse": "traces are recorded only when Langfuse environment keys are set",
}